# Week 1 — Matrix multiplication and cosine similarity

**Chapters:** Linear Algebra Ch 1 (vectors, dot product) and Ch 2 (matrix multiplication, block matrices).  
**Build goal:** write matmul and cosine similarity from the definitions, confirm they match NumPy, and rank a few "embeddings" against a query.

In [ ]:
import numpy as np
np.random.seed(0)

## 1. Matrix multiplication from the definition

$$(AB)_{ik} = \sum_j A_{ij} B_{jk}$$

In [ ]:
def matmul_loops(A, B):
    m, n = A.shape
    n2, p = B.shape
    assert n == n2, 'inner dimensions must match'
    C = np.zeros((m, p))
    for i in range(m):
        for k in range(p):
            for j in range(n):
                C[i, k] += A[i, j] * B[j, k]
    return C

def matmul_einsum(A, B):
    return np.einsum('ij,jk->ik', A, B)

A = np.random.randn(4, 3)
B = np.random.randn(3, 5)
assert np.allclose(matmul_loops(A, B), A @ B)
assert np.allclose(matmul_einsum(A, B), A @ B)
print('matmul ok, shape', (A @ B).shape)

## 2. Block matrices

Multi-head attention concatenates the outputs of $h$ heads into one matrix and applies a single output projection. Check that multiplying block-wise agrees with multiplying the concatenation.

In [ ]:
h, d_head, d_model = 4, 8, 32
heads = [np.random.randn(6, d_head) for _ in range(h)]      # 6 tokens, 4 heads
W_o = np.random.randn(d_model, d_model)

concat = np.concatenate(heads, axis=1)                      # (6, 32)
full = concat @ W_o

# same thing as a sum of block products: rows of W_o split per head
blocks = [heads[i] @ W_o[i*d_head:(i+1)*d_head, :] for i in range(h)]
assert np.allclose(full, sum(blocks))
print('block multiplication ok')

## 3. Cosine similarity

$$\cos\theta = \frac{a\cdot b}{\lVert a\rVert\,\lVert b\rVert}$$

By Cauchy–Schwarz the result lies in $[-1, 1]$.

In [ ]:
def cosine_similarity(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

a = np.random.randn(16); b = np.random.randn(16)
assert -1 <= cosine_similarity(a, b) <= 1
assert np.isclose(cosine_similarity(a, a), 1.0)
assert np.isclose(cosine_similarity(a, -a), -1.0)
assert np.isclose(cosine_similarity(a, 3 * a), 1.0)   # magnitude does not matter
print('cosine ok')

## 4. A toy retrieval step

Store normalised 'passage' vectors as rows of a matrix $E$. Then ranking against a query $q$ is a single matrix–vector product $Eq$.

In [ ]:
labels = ['fever and cough', 'referral to county hospital', 'ambulance dispatch time',
          'vaccination schedule', 'malaria test result']
E = np.random.randn(len(labels), 16)
E = E / np.linalg.norm(E, axis=1, keepdims=True)        # unit rows

q = E[2] + 0.3 * np.random.randn(16)                    # a noisy version of item 2
q = q / np.linalg.norm(q)

scores = E @ q                                           # cosine similarity for every row at once
for i in np.argsort(-scores):
    print(f'{scores[i]:+.3f}  {labels[i]}')

# confirm the fast path equals the per-pair definition
assert np.allclose(scores, [cosine_similarity(E[i], q) for i in range(len(labels))])

## 5. Where it lives in AI

- Every dense layer and every attention score is a matrix multiplication; `einsum` notation is how papers write them.
- Block structure is how multi-head attention and tensor-parallel sharding work.
- Cosine similarity over unit-normalised embeddings is the ranking step of retrieval-augmented generation, and the same dot product (without normalisation) is the attention score $QK^\top$.